In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saveded outside of the current session

# Using the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your kernel
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import torch
import os
import shutil

backbone_b14 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
print("DINOv2 ViT-B/14 téléchargé et mis en cache")

cache_dir = '/root/.cache/torch/hub'
shutil.make_archive('/kaggle/working/dinov2_vitb14_cache', 'zip', cache_dir)
print("Cache compressé:", os.path.getsize('/kaggle/working/dinov2_vitb14_cache.zip') / 1e6, "MB")

In [2]:
import os, shutil, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

FINDING_COLS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
                'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's",
                'Contusion', 'Fracture']
COMP = '/kaggle/input/competitions/rsna-knee-abnormality-detection'

# Labels
final_backup_path = '/kaggle/input/datasets/azerkh/pseudo-labels-backup/full_labels_final.csv'
final_output_path = '/kaggle/working/full_labels_final.csv'
if os.path.exists(final_backup_path) and not os.path.exists(final_output_path):
    shutil.copy(final_backup_path, final_output_path)
full_labels = pd.read_csv(final_output_path)
print("Total studies:", len(full_labels))

# DINOv2 ViT-S cache (must run before any torch.hub.load)
cache_source = '/kaggle/input/datasets/azerkh/dinov2-cache-backup'
cache_dest = '/root/.cache/torch/hub'
if os.path.exists(cache_source) and not os.path.exists(os.path.join(cache_dest, 'checkpoints')):
    os.makedirs(cache_dest, exist_ok=True)
    for sub in ['checkpoints', 'facebookresearch_dinov2_main']:
        src = os.path.join(cache_source, sub)
        if os.path.exists(src):
            shutil.copytree(src, os.path.join(cache_dest, sub), dirs_exist_ok=True)
    trusted = os.path.join(cache_source, 'trusted_list')
    if os.path.exists(trusted):
        shutil.copy(trusted, os.path.join(cache_dest, 'trusted_list'))
print("ViT-S checkpoint present:",
      os.path.exists(os.path.join(cache_dest, 'checkpoints', 'dinov2_vits14_pretrain.pth')))

Total studies: 4407
ViT-S checkpoint present: True


In [3]:
import glob, cv2, pydicom
from multiprocessing import Pool
from concurrent.futures import ThreadPoolExecutor

SIZE = 224          # 16 x 14 patches
N_WIN = 4           # windows per plane
CROP_MM = 140.0
PLANES = ['Sagittal', 'Axial', 'Coronal']

try:
    os.makedirs('/kaggle/temp', exist_ok=True)
    TMP = '/kaggle/temp'
except Exception:
    TMP = '/tmp'
CACHE_TRAIN = f'{TMP}/cache_train'
CACHE_TEST = f'{TMP}/cache_test'
os.makedirs(CACHE_TRAIN, exist_ok=True)
os.makedirs(CACHE_TEST, exist_ok=True)
print("Free disk (GB):", round(shutil.disk_usage(TMP).free / 1e9, 1))

def pick_series(plane_df):
    ranked = plane_df.sort_values(['Fluid_Sensitive', 'Fat_Suppression'],
                                  ascending=False, kind='stable')
    return ranked.iloc[0]['SeriesInstanceUID']

def read_header(path):
    try:
        h = pydicom.dcmread(path, stop_before_pixels=True)
        iop = getattr(h, 'ImageOrientationPatient', None)
        ipp = getattr(h, 'ImagePositionPatient', None)
        if iop is not None and ipp is not None and len(iop) == 6:
            r = np.array(iop[:3], float); c = np.array(iop[3:], float)
            pos = float(np.dot(np.array(ipp, float), np.cross(r, c)))
        else:
            pos = float(getattr(h, 'InstanceNumber', 0) or 0)
        ps = getattr(h, 'PixelSpacing', None)
        ps = float(ps[0]) if ps is not None else 0.5
        return pos, ps
    except Exception:
        return 0.0, 0.5

def read_pixels(path):
    d = pydicom.dcmread(path)
    a = d.pixel_array.astype(np.float32)
    if a.ndim == 3:
        a = a[0]
    if str(getattr(d, 'PhotometricInterpretation', '')) == 'MONOCHROME1':
        a = a.max() - a
    return a

def crop_resize(a, ps):
    h, w = a.shape
    cpx = min(int(round(CROP_MM / max(ps, 1e-3))), h, w)
    y0, x0 = (h - cpx) // 2, (w - cpx) // 2
    a = a[y0:y0 + cpx, x0:x0 + cpx]
    return cv2.resize(a, (SIZE, SIZE), interpolation=cv2.INTER_AREA)

def build_plane(study_id, plane, study_rows, root):
    pr = study_rows[study_rows['Anatomical_Plane'] == plane]
    if len(pr) == 0:
        return None
    sid = pick_series(pr)
    files = glob.glob(f'{root}/{study_id}/{sid}/*.dcm')
    if not files:
        return None
    with ThreadPoolExecutor(8) as ex:
        hdrs = list(ex.map(read_header, files))
    order = sorted(range(len(files)), key=lambda i: hdrs[i][0])   # anatomical order
    ordered = [files[i] for i in order]
    ps = float(np.median([h[1] for h in hdrs]))
    n = len(ordered)
    if n >= 3:
        centers = np.linspace(0.06 * (n - 1), 0.94 * (n - 1), N_WIN).round().astype(int)
        centers = np.clip(centers, 1, n - 2)
    else:
        centers = np.zeros(N_WIN, dtype=int)
    need = sorted(set(int(np.clip(i, 0, n - 1)) for c in centers for i in (c - 1, c, c + 1)))
    imgs = {}
    for i in need:
        try:
            imgs[i] = crop_resize(read_pixels(ordered[i]), ps)
        except Exception:
            imgs[i] = None
    valid = [v for v in imgs.values() if v is not None]
    if not valid:
        return None
    allpx = np.concatenate([v.ravel() for v in valid])
    lo, hi = np.percentile(allpx, [2, 98])

    def norm(v):
        if v is None:
            return np.zeros((SIZE, SIZE), np.uint8)
        return (np.clip((v - lo) / (hi - lo + 1e-6), 0, 1) * 255).astype(np.uint8)

    wins = np.zeros((N_WIN, 3, SIZE, SIZE), np.uint8)
    for k, c in enumerate(centers):
        for j, i in enumerate((c - 1, c, c + 1)):
            wins[k, j] = norm(imgs[int(np.clip(i, 0, n - 1))])
    return wins

_ROWS = None; _ROOT = None; _OUT = None

def _build_one(sid):
    out = f'{_OUT}/{sid}.npy'
    arr = np.zeros((3, N_WIN, 3, SIZE, SIZE), np.uint8)
    rows = _ROWS.get(sid)
    miss = 0
    if rows is None:
        miss = 3
    else:
        for p, plane in enumerate(PLANES):
            try:
                w = build_plane(sid, plane, rows, _ROOT)
            except Exception:
                w = None
            if w is None:
                miss += 1
            else:
                arr[p] = w
    np.save(out + '.tmp.npy', arr)
    os.replace(out + '.tmp.npy', out)
    return sid, miss

def build_cache(study_ids, series_df, root, out_dir, nproc=4):
    global _ROWS, _ROOT, _OUT
    _ROWS = {k: v for k, v in series_df.groupby('StudyInstanceUID')}
    _ROOT, _OUT = root, out_dir
    todo = [s for s in study_ids if not os.path.exists(f'{out_dir}/{s}.npy')]
    print(f"Building cache for {len(todo)} studies...")
    t0 = time.time(); miss_total = 0
    with Pool(nproc) as pool:
        for i, (_, miss) in enumerate(pool.imap_unordered(_build_one, todo, chunksize=4)):
            miss_total += miss
            if (i + 1) % 500 == 0:
                print(f"  {i+1}/{len(todo)} | {time.time()-t0:.0f}s")
    n_planes = max(1, 3 * len(todo))
    print(f"Done in {time.time()-t0:.0f}s | missing planes: {miss_total}/{n_planes} "
          f"({100*miss_total/n_planes:.1f}%)")
    if miss_total / n_planes > 0.2:
        print("WARNING: many planes failed to decode. DICOM decoders may be missing.")

train_series = pd.read_csv(f'{COMP}/train_series.csv')
build_cache(full_labels['StudyInstanceUID'].tolist(), train_series,
            f'{COMP}/train_series', CACHE_TRAIN)

Free disk (GB): 1148.7
Building cache for 4407 studies...
  500/4407 | 75s
  1000/4407 | 149s
  1500/4407 | 221s
  2000/4407 | 294s
  2500/4407 | 365s
  3000/4407 | 438s
  3500/4407 | 510s
  4000/4407 | 583s
Done in 643s | missing planes: 0/13221 (0.0%)


In [10]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset

train_raw = pd.read_csv(f'{COMP}/train.csv')
gold_ids = set(train_raw[train_raw[FINDING_COLS].notna().all(axis=1)]['StudyInstanceUID'])
train_ids = [s for s in full_labels['StudyInstanceUID'] if s not in gold_ids]
val_ids = sorted(gold_ids)

def adjust_labels(df):
    d = df.copy()
    is_half = lambda s: (s - 0.5).abs() < 1e-6
    for col in ["Baker's", 'Medial OA']:
        d.loc[is_half(d[col]), col] = 0.05
    syn_silent = is_half(d['Synovitis'])
    d.loc[syn_silent & (d['Effusion'] > 0.5), 'Synovitis'] = 0.60
    d.loc[syn_silent & (d['Effusion'] < 0.5), 'Synovitis'] = 0.25
    for col in FINDING_COLS:
        d.loc[is_half(d[col]), col] = -1
    return d

LABEL_MODE = 'adjusted'   # change to 'adjusted' for the next experiment, one change at a time
labels_for_train = adjust_labels(full_labels) if LABEL_MODE == 'adjusted' else full_labels

class KneeCacheDataset(Dataset):
    def __init__(self, ids, cache_dir, labels_df=None):
        self.ids = list(ids)
        self.cache_dir = cache_dir
        self.labels = None
        if labels_df is not None:
            L = labels_df.set_index('StudyInstanceUID').loc[self.ids, FINDING_COLS]
            self.labels = L.astype(np.float32).fillna(0.5).values

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        arr = np.load(f'{self.cache_dir}/{self.ids[i]}.npy')
        mask = arr.reshape(3, -1).max(1) > 0
        y = self.labels[i] if self.labels is not None else np.zeros(len(FINDING_COLS), np.float32)
        return torch.from_numpy(arr), torch.from_numpy(mask), torch.from_numpy(y)

train_ds = KneeCacheDataset(train_ids, CACHE_TRAIN, labels_for_train)
val_ds = KneeCacheDataset(val_ids, CACHE_TRAIN, full_labels)   # gold labels untouched
print("Train:", len(train_ds), "| Val (gold):", len(val_ds), "| Label mode:", LABEL_MODE)

Train: 4349 | Val (gold): 58 | Label mode: adjusted


In [15]:
from sklearn.model_selection import KFold

QUICK = False   # True: 1 fold, 3 epochs (test). False: 5 folds, 6 epochs (final commit)
N_FOLDS = 5
FOLDS_TO_RUN = [0] if QUICK else list(range(N_FOLDS))
EPOCHS = 3 if QUICK else 6
BATCH = 8

device = torch.device('cuda')

class KneeNet25D(nn.Module):
    def __init__(self, n_findings=12, dim=256, n_unfrozen=2):
        super().__init__()
        self.backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
        for p in self.backbone.parameters():
            p.requires_grad = False
        nb = len(self.backbone.blocks)
        for blk in self.backbone.blocks[nb - n_unfrozen:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in self.backbone.norm.parameters():
            p.requires_grad = True
        self.drop = nn.Dropout(0.2)
        self.proj = nn.Sequential(nn.Linear(2 * 384, dim), nn.LayerNorm(dim))
        self.plane_emb = nn.Embedding(3, dim)
        self.att = nn.Sequential(nn.Linear(dim, 128), nn.Tanh(), nn.Dropout(0.2),
                                 nn.Linear(128, n_findings))
        self.cls_w = nn.Parameter(torch.randn(n_findings, dim) * 0.02)
        self.cls_b = nn.Parameter(torch.zeros(n_findings))
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x, mask):
        B, P, W = x.shape[:3]                                  # x: uint8 (B,P,W,3,S,S)
        xf = x.flatten(0, 2).float().div(255.0)
        xf = (xf - self.mean) / self.std
        out = self.backbone.forward_features(xf)
        f = torch.cat([out['x_norm_clstoken'], out['x_norm_patchtokens'].mean(1)], dim=-1)
        h = self.proj(self.drop(f)).view(B, P, W, -1)
        h = h + self.plane_emb.weight.view(1, P, 1, -1)
        h = h.reshape(B, P * W, -1)
        pad = ~mask.unsqueeze(-1).expand(B, P, W).reshape(B, P * W)
        pad = pad & ~pad.all(1, keepdim=True)                  # study with no planes: attend to all
        a = self.att(h).float().masked_fill(pad.unsqueeze(-1), -1e4).softmax(1)   # (B,T,12)
        pooled = torch.einsum('btc,btd->bcd', a.to(h.dtype), h)
        return ((pooled * self.cls_w).sum(-1) + self.cls_b).float()

def masked_bce_loss(outputs, labels):
    mask = (labels != -1).float()
    lab = labels.clone()
    lab[labels == -1] = 0
    loss = nn.BCEWithLogitsLoss(reduction='none')(outputs, lab) * mask
    return loss.sum() / mask.sum().clamp(min=1)

@torch.no_grad()
def predict_ensemble(models, loader):
    for m in models:
        m.eval()
    chunks = []
    for x, mk, _ in loader:
        x, mk = x.to(device, non_blocking=True), mk.to(device)
        ps = []
        for m in models:
            with torch.autocast('cuda', dtype=torch.float16):
                ps.append(torch.sigmoid(m(x, mk)).float().cpu().numpy())
        chunks.append(np.stack(ps))
    return np.concatenate(chunks, axis=1)                      # (n_models, N, 12)

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
splits = list(kf.split(np.arange(len(train_ds))))
models_ensemble = []

for fold in FOLDS_TO_RUN:
    tr_idx, _ = splits[fold]
    print(f"\n===== FOLD {fold+1}/{N_FOLDS} | {len(tr_idx)} studies =====")
    loader = DataLoader(Subset(train_ds, tr_idx), batch_size=BATCH, shuffle=True,
                        num_workers=2, pin_memory=True, drop_last=True)
    model = KneeNet25D().to(device)
    bb = [p for p in model.backbone.parameters() if p.requires_grad]
    head = [p for n, p in model.named_parameters()
            if not n.startswith('backbone.') and p.requires_grad]
    opt = torch.optim.AdamW([{'params': bb, 'lr': 1e-5}, {'params': head, 'lr': 1e-3}],
                            weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = torch.amp.GradScaler('cuda')
    for epoch in range(EPOCHS):
        model.train()
        t0 = time.time(); total = 0.0
        for x, mk, y in loader:
            x, mk, y = x.to(device, non_blocking=True), mk.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16):
                out = model(x, mk)
            loss = masked_bce_loss(out.float(), y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(bb + head, 1.0)
            scaler.step(opt)
            scaler.update()
            total += loss.item()
        sched.step()
        print(f"  Fold {fold+1} | Epoch {epoch+1}/{EPOCHS} | loss {total/len(loader):.4f} "
              f"| {time.time()-t0:.0f}s")
    models_ensemble.append(model)
    torch.cuda.empty_cache()

print(f"\n{len(models_ensemble)} model(s) trained")


===== FOLD 1/5 | 3479 studies =====


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


  Fold 1 | Epoch 1/6 | loss 0.6034 | 87s
  Fold 1 | Epoch 2/6 | loss 0.5773 | 86s
  Fold 1 | Epoch 3/6 | loss 0.5609 | 85s
  Fold 1 | Epoch 4/6 | loss 0.5467 | 86s
  Fold 1 | Epoch 5/6 | loss 0.5336 | 86s
  Fold 1 | Epoch 6/6 | loss 0.5229 | 86s

===== FOLD 2/5 | 3479 studies =====


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


  Fold 2 | Epoch 1/6 | loss 0.6069 | 87s
  Fold 2 | Epoch 2/6 | loss 0.5800 | 86s
  Fold 2 | Epoch 3/6 | loss 0.5654 | 85s
  Fold 2 | Epoch 4/6 | loss 0.5496 | 86s
  Fold 2 | Epoch 5/6 | loss 0.5359 | 86s
  Fold 2 | Epoch 6/6 | loss 0.5258 | 86s

===== FOLD 3/5 | 3479 studies =====


...

  Fold 3 | Epoch 1/6 | loss 0.6053 | 87s
  Fold 3 | Epoch 2/6 | loss 0.5783 | 86s
  Fold 3 | Epoch 3/6 | loss 0.5649 | 86s
  Fold 3 | Epoch 4/6 | loss 0.5502 | 86s
  Fold 3 | Epoch 5/6 | loss 0.5343 | 86s
  Fold 3 | Epoch 6/6 | loss 0.5235 | 86s

===== FOLD 4/5 | 3479 studies =====


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


  Fold 4 | Epoch 1/6 | loss 0.6073 | 87s
  Fold 4 | Epoch 2/6 | loss 0.5813 | 86s
  Fold 4 | Epoch 3/6 | loss 0.5650 | 86s
  Fold 4 | Epoch 4/6 | loss 0.5522 | 87s
  Fold 4 | Epoch 5/6 | loss 0.5381 | 86s
  Fold 4 | Epoch 6/6 | loss 0.5279 | 86s

===== FOLD 5/5 | 3480 studies =====


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


  Fold 5 | Epoch 1/6 | loss 0.6063 | 87s
  Fold 5 | Epoch 2/6 | loss 0.5784 | 86s
  Fold 5 | Epoch 3/6 | loss 0.5609 | 86s
  Fold 5 | Epoch 4/6 | loss 0.5470 | 86s
  Fold 5 | Epoch 5/6 | loss 0.5333 | 86s
  Fold 5 | Epoch 6/6 | loss 0.5222 | 86s

5 model(s) trained


In [16]:
from sklearn.model_selection import KFold

QUICK = False   # True: 1 fold, 3 epochs (test). False: 5 folds, 6 epochs (final commit)
N_FOLDS = 5
FOLDS_TO_RUN = [0] if QUICK else list(range(N_FOLDS))
EPOCHS = 3 if QUICK else 6
BATCH = 8

device = torch.device('cuda')

class KneeNet25D(nn.Module):
    def __init__(self, n_findings=12, dim=256, n_unfrozen=2):
        super().__init__()
        self.backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
        for p in self.backbone.parameters():
            p.requires_grad = False
        nb = len(self.backbone.blocks)
        for blk in self.backbone.blocks[nb - n_unfrozen:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in self.backbone.norm.parameters():
            p.requires_grad = True
        self.drop = nn.Dropout(0.2)
        self.proj = nn.Sequential(nn.Linear(2 * 384, dim), nn.LayerNorm(dim))
        self.plane_emb = nn.Embedding(3, dim)
        self.att = nn.Sequential(nn.Linear(dim, 128), nn.Tanh(), nn.Dropout(0.2),
                                 nn.Linear(128, n_findings))
        self.cls_w = nn.Parameter(torch.randn(n_findings, dim) * 0.02)
        self.cls_b = nn.Parameter(torch.zeros(n_findings))
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x, mask):
        B, P, W = x.shape[:3]                                  # x: uint8 (B,P,W,3,S,S)
        xf = x.flatten(0, 2).float().div(255.0)
        xf = (xf - self.mean) / self.std
        out = self.backbone.forward_features(xf)
        f = torch.cat([out['x_norm_clstoken'], out['x_norm_patchtokens'].mean(1)], dim=-1)
        h = self.proj(self.drop(f)).view(B, P, W, -1)
        h = h + self.plane_emb.weight.view(1, P, 1, -1)
        h = h.reshape(B, P * W, -1)
        pad = ~mask.unsqueeze(-1).expand(B, P, W).reshape(B, P * W)
        pad = pad & ~pad.all(1, keepdim=True)                  # study with no planes: attend to all
        a = self.att(h).float().masked_fill(pad.unsqueeze(-1), -1e4).softmax(1)   # (B,T,12)
        pooled = torch.einsum('btc,btd->bcd', a.to(h.dtype), h)
        return ((pooled * self.cls_w).sum(-1) + self.cls_b).float()

def masked_bce_loss(outputs, labels):
    mask = (labels != -1).float()
    lab = labels.clone()
    lab[labels == -1] = 0
    loss = nn.BCEWithLogitsLoss(reduction='none')(outputs, lab) * mask
    return loss.sum() / mask.sum().clamp(min=1)

@torch.no_grad()
def predict_ensemble(models, loader):
    for m in models:
        m.eval()
    chunks = []
    for x, mk, _ in loader:
        x, mk = x.to(device, non_blocking=True), mk.to(device)
        ps = []
        for m in models:
            with torch.autocast('cuda', dtype=torch.float16):
                ps.append(torch.sigmoid(m(x, mk)).float().cpu().numpy())
        chunks.append(np.stack(ps))
    return np.concatenate(chunks, axis=1)                      # (n_models, N, 12)

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
splits = list(kf.split(np.arange(len(train_ds))))
models_ensemble = []

for fold in FOLDS_TO_RUN:
    tr_idx, _ = splits[fold]
    print(f"\n===== FOLD {fold+1}/{N_FOLDS} | {len(tr_idx)} studies =====")
    loader = DataLoader(Subset(train_ds, tr_idx), batch_size=BATCH, shuffle=True,
                        num_workers=2, pin_memory=True, drop_last=True)
    model = KneeNet25D().to(device)
    bb = [p for p in model.backbone.parameters() if p.requires_grad]
    head = [p for n, p in model.named_parameters()
            if not n.startswith('backbone.') and p.requires_grad]
    opt = torch.optim.AdamW([{'params': bb, 'lr': 1e-5}, {'params': head, 'lr': 1e-3}],
                            weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = torch.amp.GradScaler('cuda')
    for epoch in range(EPOCHS):
        model.train()
        t0 = time.time(); total = 0.0
        for x, mk, y in loader:
            x, mk, y = x.to(device, non_blocking=True), mk.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16):
                out = model(x, mk)
            loss = masked_bce_loss(out.float(), y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(bb + head, 1.0)
            scaler.step(opt)
            scaler.update()
            total += loss.item()
        sched.step()
        print(f"  Fold {fold+1} | Epoch {epoch+1}/{EPOCHS} | loss {total/len(loader):.4f} "
              f"| {time.time()-t0:.0f}s")
    models_ensemble.append(model)
    torch.cuda.empty_cache()

print(f"\n{len(models_ensemble)} model(s) trained")


===== FOLD 1/5 | 3479 studies =====


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


  Fold 1 | Epoch 1/6 | loss 0.6041 | 87s
  Fold 1 | Epoch 2/6 | loss 0.5783 | 86s
  Fold 1 | Epoch 3/6 | loss 0.5631 | 86s
  Fold 1 | Epoch 4/6 | loss 0.5498 | 86s
  Fold 1 | Epoch 5/6 | loss 0.5362 | 86s
  Fold 1 | Epoch 6/6 | loss 0.5254 | 86s

===== FOLD 2/5 | 3479 studies =====


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


  Fold 2 | Epoch 1/6 | loss 0.6080 | 87s
  Fold 2 | Epoch 2/6 | loss 0.5821 | 86s
  Fold 2 | Epoch 3/6 | loss 0.5658 | 86s
  Fold 2 | Epoch 4/6 | loss 0.5507 | 86s
  Fold 2 | Epoch 5/6 | loss 0.5382 | 87s


...

  Fold 3 | Epoch 1/6 | loss 0.6057 | 87s
  Fold 3 | Epoch 2/6 | loss 0.5792 | 86s
  Fold 3 | Epoch 3/6 | loss 0.5617 | 86s
  Fold 3 | Epoch 4/6 | loss 0.5460 | 87s
  Fold 3 | Epoch 5/6 | loss 0.5329 | 86s
  Fold 3 | Epoch 6/6 | loss 0.5231 | 86s

===== FOLD 4/5 | 3479 studies =====


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


  Fold 4 | Epoch 1/6 | loss 0.6059 | 87s
  Fold 4 | Epoch 2/6 | loss 0.5787 | 86s
  Fold 4 | Epoch 3/6 | loss 0.5635 | 86s
  Fold 4 | Epoch 4/6 | loss 0.5502 | 86s
  Fold 4 | Epoch 5/6 | loss 0.5364 | 86s
  Fold 4 | Epoch 6/6 | loss 0.5259 | 86s

===== FOLD 5/5 | 3480 studies =====


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


  Fold 5 | Epoch 1/6 | loss 0.6068 | 87s
  Fold 5 | Epoch 2/6 | loss 0.5809 | 86s
  Fold 5 | Epoch 3/6 | loss 0.5629 | 87s
  Fold 5 | Epoch 4/6 | loss 0.5468 | 86s
  Fold 5 | Epoch 5/6 | loss 0.5336 | 86s
  Fold 5 | Epoch 6/6 | loss 0.5241 | 86s

5 model(s) trained


In [17]:
from sklearn.metrics import roc_auc_score

val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2)
preds = predict_ensemble(models_ensemble, val_loader)          # (M, 58, 12)
y_val = val_ds.labels

def macro_auc(y, p, verbose=False):
    aucs = {}
    for i, c in enumerate(FINDING_COLS):
        if len(np.unique(y[:, i])) < 2:
            continue
        aucs[c] = roc_auc_score(y[:, i], p[:, i])
        if verbose:
            print(f"{c}: {aucs[c]:.4f}")
    return float(np.mean(list(aucs.values())))

for k in range(preds.shape[0]):
    print(f"Model {k+1} macro AUC: {macro_auc(y_val, preds[k]):.4f}")
print("\nEnsemble per finding:")
print(f"Ensemble macro AUC: {macro_auc(y_val, preds.mean(0), verbose=True):.4f}")

Model 1 macro AUC: 0.7580
Model 2 macro AUC: 0.7635
Model 3 macro AUC: 0.7742
Model 4 macro AUC: 0.7605
Model 5 macro AUC: 0.7604

Ensemble per finding:
ACL: 0.6311
MCL: 0.6508
Medial Meniscus: 0.6863
Lateral Meniscus: 0.8099
Medial OA: 0.9271
Lateral OA: 0.7466
PF OA: 0.7683
Effusion: 0.9292
Synovitis: 0.7634
Baker's: 0.9112
Contusion: 0.6937
Fracture: 0.7458
Ensemble macro AUC: 0.7720


In [19]:
test = pd.read_csv(f'{COMP}/test.csv')
test_series = pd.read_csv(f'{COMP}/test_series.csv')
sample_sub = pd.read_csv(f'{COMP}/sample_submission.csv')
test_ids = test['StudyInstanceUID'].tolist()

build_cache(test_ids, test_series, f'{COMP}/test_series', CACHE_TEST)

test_ds = KneeCacheDataset(test_ids, CACHE_TEST, None)
test_loader = DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=2)
test_preds = predict_ensemble(models_ensemble, test_loader).mean(0)   # (N, 12)

pred_df = pd.DataFrame(test_preds, columns=FINDING_COLS)
pred_df['StudyInstanceUID'] = test_ids
submission = sample_sub[['StudyInstanceUID']].merge(pred_df, on='StudyInstanceUID', how='left')
submission = submission[sample_sub.columns]

assert submission.shape == sample_sub.shape
assert submission['StudyInstanceUID'].equals(sample_sub['StudyInstanceUID'])
assert submission[FINDING_COLS].isna().sum().sum() == 0

submission.to_csv('/kaggle/working/submission.csv', index=False)
print("submission.csv created")
print(submission.head())

Building cache for 0 studies...
Done in 0s | missing planes: 0/1 (0.0%)
submission.csv created
                                    StudyInstanceUID       ACL       MCL  \
0  1.2.826.0.1.3680043.8.498.10047035057544427318...  0.174502  0.160684   
1  1.2.826.0.1.3680043.8.498.10062861783145312629...  0.507643  0.313210   
2  1.2.826.0.1.3680043.8.498.10067514707072572280...  0.095015  0.181396   

   Medial Meniscus  Lateral Meniscus  Medial OA  Lateral OA     PF OA  \
0         0.450462          0.208295   0.507467    0.614278  0.628013   
1         0.388284          0.341230   0.272199    0.803307  0.864059   
2         0.589885          0.155863   0.353158    0.332666  0.656388   

   Effusion  Synovitis   Baker's  Contusion  Fracture  
0  0.649622   0.561249  0.226891   0.641826  0.235700  
1  0.725707   0.689136  0.314700   0.583746  0.463628  
2  0.417568   0.358878  0.182584   0.378800  0.236755  
